# 🔮 FactSphere — Groq Cloud Edition
**Runs fully in Google Colab — no GPU or local hardware needed!**

### Steps:
1. Run **Cell 1** to install packages
2. Run **Cell 2** to write the app
3. Run **Cell 3** — paste your Groq API key when prompted, then open the public URL

> 🔑 Get a **free** Groq API key at [console.groq.com](https://console.groq.com)

In [ ]:
# ── Cell 1: Install all dependencies ──────────────────────────────────────────
!pip install -q streamlit groq pyngrok sentence-transformers \
                chromadb scikit-learn wikipedia-api numpy

In [ ]:
# ── Cell 2: Write app.py ───────────────────────────────────────────────────────
app_code = '''
import streamlit as st
import numpy as np
import wikipediaapi
import chromadb
import json
import re
import os
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from groq import Groq

st.set_page_config(page_title="FactSphere QA", page_icon="🔮", layout="wide")

st.markdown("""
<style>
@import url(\'https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap\');
html, body, [class*="css"] { font-family: \'Inter\', sans-serif; }
.main-header {
    background: linear-gradient(135deg,#1a1a2e,#16213e,#0f3460);
    padding:2rem 2.5rem; border-radius:16px; margin-bottom:2rem;
    text-align:center; box-shadow:0 8px 32px rgba(233,69,96,.15);
}
.main-header h1 { color:#e94560; font-size:2.8rem; margin:0; letter-spacing:-1px; }
.main-header p  { color:#a8b2d8; margin:.5rem 0 0; font-size:1rem; }
.result-card {
    background:#0d0d1a; border:1px solid #1e2a45; border-radius:14px;
    padding:1.5rem; margin-bottom:1.5rem; box-shadow:0 4px 16px rgba(0,0,0,.4);
}
.q-label { color:#a8b2d8; font-size:.8rem; font-weight:600;
           text-transform:uppercase; letter-spacing:1px; margin-bottom:4px; }
.q-text  { color:#e2e8f0; font-size:1.1rem; font-weight:600; margin-bottom:1rem; }
.verdict-answer  { background:#0d2318; border-left:4px solid #2ecc71;
                   padding:1rem 1.2rem; border-radius:8px; color:#c6f6d5; }
.verdict-clarify { background:#2a2000; border-left:4px solid #f39c12;
                   padding:1rem 1.2rem; border-radius:8px; color:#fefcbf; }
.verdict-refuse  { background:#2a0a0a; border-left:4px solid #e74c3c;
                   padding:1rem 1.2rem; border-radius:8px; color:#fed7d7; }
.badge { display:inline-block; padding:.2rem .7rem; border-radius:20px;
         font-size:.78rem; font-weight:700; margin-right:6px; }
.badge-factual     { background:#0f3460; color:#90cdf4; }
.badge-speculative { background:#3d1f00; color:#fbd38d; }
.badge-ambiguous   { background:#2d1558; color:#d6bcfa; }
.badge-supported   { background:#0d2318; color:#68d391; }
.badge-unsupported { background:#2a0a0a; color:#fc8181; }
hr.div { border:none; border-top:1px solid #1e2a45; margin:2rem 0; }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="main-header">
  <h1>🔮 FactSphere</h1>
  <p>Hallucination-Aware Multi-Agent QA · Powered by Groq Cloud ⚡ (Free)</p>
</div>
""", unsafe_allow_html=True)

# ── Sidebar ──
st.sidebar.markdown("## ⚙️ Configuration")
GROQ_API_KEY = st.sidebar.text_input("🔑 Groq API Key", type="password",
    value=os.environ.get(\'GROQ_API_KEY\', \'\'),
    placeholder="gsk_...",
    help="Get free key at https://console.groq.com")
GROQ_MODEL = st.sidebar.selectbox("🤖 Model",
    ["llama3-8b-8192","llama3-70b-8192","mixtral-8x7b-32768","gemma2-9b-it"])
st.sidebar.info("**Free Groq setup:**\n1. [console.groq.com](https://console.groq.com)\n2. Sign up → API Keys → Create\n3. Paste above")
if st.sidebar.button("🗑️ Clear History"):
    st.session_state.history = []
    st.rerun()
st.sidebar.markdown("---")
st.sidebar.markdown("""**🔁 Pipeline**\n1. 🧠 Planner\n2. 📚 Retriever\n3. ✍️ Generator\n4. ✅ Verifier\n5. 🎯 Confidence""")

def groq_call(api_key, model, system, user, as_json=False):
    client = Groq(api_key=api_key)
    kwargs = dict(model=model,
        messages=[("role","system"),("role","user")],
        max_tokens=400, temperature=0)
    kwargs["messages"] = [
        {"role":"system","content":system},
        {"role":"user",  "content":user}
    ]
    if as_json:
        kwargs["response_format"] = {"type":"json_object"}
    resp = client.chat.completions.create(**kwargs)
    return resp.choices[0].message.content.strip()

def safe_json(text, keys, defaults):
    for src in [text, re.search(r\"\\{.*?\\}\", text, re.DOTALL)]:
        try:
            raw  = src if isinstance(src,str) else src.group()
            data = json.loads(raw)
            return {k: data.get(k, defaults[k]) for k in keys}
        except: pass
    return defaults

@st.cache_resource
def load_resources():
    embedder = SentenceTransformer(\'all-MiniLM-L6-v2\')
    wiki     = wikipediaapi.Wikipedia(user_agent=\'FactSphere/1.0\', language=\'en\')
    chroma   = chromadb.Client()
    docs = [
        "Machine learning is a subset of AI that builds systems learning from data.",
        "Neural networks mimic the human brain to recognize patterns.",
        "Large Language Models (LLMs) are trained on vast text data.",
        "Albert Einstein developed the theory of relativity.",
        "Marie Curie pioneered research on radioactivity and won Nobel Prizes.",
        "Nikola Tesla designed the modern AC electricity supply system.",
        "The Industrial Revolution occurred roughly between 1760 and 1840.",
        "The Great Wall of China protected ancient Chinese states.",
        "Mount Everest is Earth highest mountain above sea level.",
        "The Amazon River is the largest river by discharge volume.",
    ]
    try:
        col = chroma.create_collection("factsphere")
        col.add(embeddings=embedder.encode(docs).tolist(), documents=docs,
                ids=[f"d{i}" for i in range(len(docs))])
    except:
        col = chroma.get_collection("factsphere")
    return embedder, wiki, col

def run_planner(query, api_key, model):
    try:
        raw = groq_call(api_key, model,
            \'Classify query as factual/speculative/ambiguous. Return ONLY JSON: {"query_type":"factual","reasoning":"..."}\',
            f"Query: {query}", as_json=True)
        return safe_json(raw,["query_type","reasoning"],["ambiguous","N/A"] and {"query_type":"ambiguous","reasoning":"N/A"})
    except Exception as e:
        return {"query_type":"ambiguous","reasoning":str(e)}

def run_retriever(query, embedder, wiki, collection):
    qemb   = embedder.encode(query).tolist()
    res    = collection.query(query_embeddings=[qemb], n_results=3)
    chunks = res["documents"][0] if res["documents"] else []
    sims   = []
    if chunks:
        cembs = embedder.encode(chunks)
        sims  = cosine_similarity([qemb], cembs)[0].tolist()
    best = max(sims) if sims else 0.0
    if best < 0.4:
        page = wiki.page(query)
        if page.exists():
            chunks.append("Wikipedia: " + page.summary[:500])
            sims.append(best)
    return chunks, sims

def run_generator(query, chunks, api_key, model):
    ctx = "\\n\\n".join(chunks) if chunks else "No context."
    try:
        return groq_call(api_key, model,
            "Answer using ONLY the context. Be concise. If unsupported say: UNSUPPORTED",
            f"Context:\\n{ctx}\\n\\nQuestion: {query}")
    except Exception as e:
        return f"UNSUPPORTED ({e})"

def run_verifier(answer, chunks, embedder, api_key, model):
    ctx = "\\n\\n".join(chunks) if chunks else ""
    try:
        raw = groq_call(api_key, model,
            \'Check if answer is supported. Return ONLY JSON: {"verdict":"SUPPORTED","reason":"..."}\',
            f"Context:\\n{ctx}\\n\\nAnswer: {answer}", as_json=True)
        p = safe_json(raw,["verdict","reason"],["NOT_SUPPORTED","Parse failed."] and {"verdict":"NOT_SUPPORTED","reason":"Parse failed."})
        verdict, reason = p["verdict"].upper(), p["reason"]
    except Exception as e:
        verdict, reason = "NOT_SUPPORTED", str(e)
    aemb    = embedder.encode([answer])
    cembs   = embedder.encode(chunks) if chunks else np.array([])
    avg_sim = float(np.mean(cosine_similarity(aemb,cembs)[0])) if len(cembs)>0 else 0.0
    if avg_sim < 0.35: verdict = "NOT_SUPPORTED"
    return verdict, reason, avg_sim

def run_confidence(q_type, verdict, iterations):
    score = 1.0
    if q_type=="speculative": score -= 0.3
    elif q_type=="ambiguous":  score -= 0.2
    if verdict=="NOT_SUPPORTED": score -= 0.3
    if iterations > 1: score -= 0.1*(iterations-1)
    score = max(0.0, min(1.0, score))
    if score>=0.6:   return score, "ANSWER",  None
    elif score>=0.3: return score, "CLARIFY", "Please clarify your query."
    else:            return score, "REFUSE",  "No reliable evidence found."

if "history" not in st.session_state:
    st.session_state.history = []

if not GROQ_API_KEY:
    st.info("👈 Enter your free Groq API key in the sidebar to start.")
    st.stop()

c1, c2 = st.columns([4,1])
with c1:
    query = st.text_input("💬 Enter your question:", placeholder="e.g. Who was Marie Curie?")
with c2:
    st.write(""); st.write("")
    submit = st.button("🚀 Ask FactSphere", use_container_width=True)

if submit and query.strip():
    embedder, wiki, collection = load_resources()
    with st.status("🤖 Running 5 agents...", expanded=True) as status:
        try:
            st.write("🧠 Agent 1/5 — Planner...")
            plan      = run_planner(query, GROQ_API_KEY, GROQ_MODEL)
            q_type    = plan["query_type"].lower().strip()
            reasoning = plan["reasoning"]
            st.write(f"   ✔ Type: **{q_type}**")

            st.write("📚 Agent 2/5 — Retriever...")
            iterations, chunks, sims = 0, [], []
            if q_type not in ["speculative","ambiguous"]:
                chunks, sims = run_retriever(query, embedder, wiki, collection)
                iterations   = 1
            st.write(f"   ✔ {len(chunks)} chunk(s)")

            st.write("✍️ Agent 3/5 — Generator...")
            answer = run_generator(query, chunks, GROQ_API_KEY, GROQ_MODEL)
            st.write("   ✔ Answer ready")

            st.write("✅ Agent 4/5 — Verifier...")
            verdict, ver_reason, avg_sim = run_verifier(answer, chunks, embedder, GROQ_API_KEY, GROQ_MODEL)
            st.write(f"   ✔ Verdict: **{verdict}**")

            if verdict=="NOT_SUPPORTED" and iterations<2:
                st.write("   ↩️ Retrying...")
                c2_, s2_ = run_retriever(query+" explanation", embedder, wiki, collection)
                chunks   = list(dict.fromkeys(chunks+c2_))
                sims     = (sims+s2_)[:len(chunks)]
                answer   = run_generator(query,chunks,GROQ_API_KEY,GROQ_MODEL)
                verdict, ver_reason, avg_sim = run_verifier(answer,chunks,embedder,GROQ_API_KEY,GROQ_MODEL)
                iterations += 1

            st.write("🎯 Agent 5/5 — Confidence...")
            score, decision, fallback = run_confidence(q_type, verdict, iterations)
            final_response = answer if decision=="ANSWER" else fallback
            st.write(f"   ✔ **{score:.0%}** | **{decision}**")
            status.update(label="✅ Done!", state="complete", expanded=False)

            st.session_state.history.insert(0,{
                "query":query,"model":GROQ_MODEL,"query_type":q_type,
                "reasoning":reasoning,"chunks":chunks,"sims":sims,
                "answer":answer,"verdict":verdict,"ver_reason":ver_reason,
                "avg_sim":avg_sim,"iterations":iterations,
                "score":score,"decision":decision,"final_response":final_response
            })
        except Exception as e:
            status.update(label="❌ Error", state="error")
            st.error(f"❌ {e}")
elif submit:
    st.warning("Please enter a question.")

if st.session_state.history:
    n = len(st.session_state.history)
    st.markdown(f"### 📋 Results — {n} {'query' if n==1 else 'queries'}")
    for idx, r in enumerate(st.session_state.history):
        qt   = r[\'query_type\']
        qcls = f"badge-{qt}" if qt in [\'factual\',\'speculative\',\'ambiguous\'] else "badge-factual"
        vcls = "badge-supported" if r[\'verdict\']=="SUPPORTED" else "badge-unsupported"
        st.markdown(f\'\'\'<div class="result-card"><div class="q-label">Query #{n-idx}</div><div class="q-text">💬 {r["query"]}</div><span class="badge {qcls}">{qt.capitalize()}</span><span class="badge {vcls}">{r["verdict"]}</span><span class="badge" style="background:#1a1a3e;color:#90cdf4">⚡ {r["model"]}</span></div>\'\'\', unsafe_allow_html=True)
        cc1,cc2,cc3,cc4 = st.columns(4)
        cc1.metric("Type",      qt.capitalize())
        cc2.metric("Verdict",   r[\'verdict\'])
        cc3.metric("Iters",     r[\'iterations\'])
        cc4.metric("Decision",  r[\'decision\'])
        st.write(f"**🎯 Confidence: {r[\'score\']:.0%}**")
        st.progress(r[\'score\'])
        d,t = r[\'decision\'], r[\'final_response\']
        if d=="ANSWER":   st.markdown(f\'<div class="verdict-answer">✅ <strong>Answer</strong><br><br>{t}</div>\', unsafe_allow_html=True)
        elif d=="CLARIFY":st.markdown(f\'<div class="verdict-clarify">⚠️ <strong>Clarify</strong><br><br>{t}</div>\', unsafe_allow_html=True)
        else:             st.markdown(f\'<div class="verdict-refuse">❌ <strong>Refused</strong><br><br>{t}</div>\', unsafe_allow_html=True)
        ca,cb = st.columns(2)
        with ca:
            with st.expander("📚 Chunks"):
                for i,ch in enumerate(r[\'chunks\']):
                    sim = r[\'sims\'][i] if i<len(r[\'sims\']) else None
                    st.markdown(f"**Chunk {i+1}**"+(f" sim={sim:.3f}" if sim else ""))
                    st.write(ch)
                    if i<len(r[\'chunks\'])-1: st.divider()
        with cb:
            with st.expander("🔍 Details"):
                st.write(f"Verdict: {r[\'verdict\']}")
                st.write(f"Reason: {r[\'ver_reason\']}")
                st.write(f"Avg Sim: {r[\'avg_sim\']:.3f}")
                st.write(f"Planner: {r[\'reasoning\']}")
        if idx<n-1: st.markdown(\'<hr class="div">\', unsafe_allow_html=True)
'''

with open('app.py', 'w') as f:
    f.write(app_code)
print('✅ app.py written!')

In [ ]:
# ── Cell 3: Launch Streamlit with public URL via ngrok ─────────────────────────
import os
from pyngrok import ngrok, conf
import threading
import time

# ── Enter your Groq API key here ──
GROQ_API_KEY = input('Paste your Groq API key (gsk_...): ').strip()
os.environ['GROQ_API_KEY'] = GROQ_API_KEY

# Kill any existing tunnels
ngrok.kill()

# Open ngrok tunnel on port 8501
public_url = ngrok.connect(8501)
print('\n' + '='*60)
print(f'🌐  Open your FactSphere app here:')
print(f'    {public_url}')
print('='*60)
print('\n⏳ Starting Streamlit... (wait 5 seconds then open the URL above)')

# Run streamlit in background
os.system('pkill -f streamlit 2>/dev/null; sleep 1')
threading.Thread(
    target=lambda: os.system(
        f'GROQ_API_KEY={GROQ_API_KEY} streamlit run app.py '
        '--server.port 8501 --server.headless true '
        '--server.enableCORS false --server.enableXsrfProtection false'
    ), daemon=True
).start()

time.sleep(5)
print(f'\n✅ Ready! Open: {public_url}')